In [1]:
import asyncio
from playwright.async_api import async_playwright
import json
import re

# We'll use this list to store our results
scraped_data = []

# Base URL for building full links
BASE_URL = "https://site.financialmodelingprep.com"

In [ ]:
# # Install the browser binaries
# !playwright install chromium

# # If you are on Linux (which your file path suggests), 
# # you might also need the system dependencies:
# !playwright install-deps chromium

170.4 MiB [                    ] 0% 277.4s170.4 MiB [                    ] 0% 137.9s170.4 MiB [                    ] 0% 102.4s170.4 MiB [                    ] 0% 85.3s170.4 MiB [                    ] 0% 73.8s170.4 MiB [                    ] 0% 69.1s170.4 MiB [                    ] 0% 54.1s170.4 MiB [                    ] 0% 43.2s170.4 MiB [                    ] 0% 34.0s170.4 MiB [                    ] 0% 25.2s170.4 MiB [                    ] 0% 21.7s170.4 MiB [                    ] 0% 19.5s170.4 MiB [                    ] 1% 15.6s170.4 MiB [                    ] 1% 12.5s170.4 MiB [                    ] 2% 10.1s170.4 MiB [=                   ] 2% 8.9s170.4 MiB [=                   ] 3% 8.0s170.4 MiB [=                   ] 3% 7.0s170.4 MiB [=                   ] 4% 6.3s170.4 MiB [=                   ] 5% 5.9s170.4 MiB [=                   ] 5% 5.6s170.4 MiB [=                   ] 6% 5.1s170.4 MiB [=                   ] 7% 4.8s170.4 MiB [==                  ] 7% 4.5s170.4 MiB [==         

In [2]:
async def get_endpoint_list():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(f"{BASE_URL}/developer/docs")
        
        # Wait for the links to be visible
        await page.wait_for_selector('a[href*="/developer/docs/stable/"]')
        
        # Extract all matching links
        links = await page.query_selector_all('a[href*="/developer/docs/stable/"]')
        
        endpoints = []
        for link in links:
            href = await link.get_attribute('href')
            text = await link.inner_text()
            if href and "stable" in href:
                endpoints.append({
                    "api_name": text.strip(),
                    "full_url": f"{BASE_URL}{href}" if href.startswith('/') else href
                })
        
        await browser.close()
        return endpoints

# Execute and get the list
endpoints = await get_endpoint_list()
print(f"Found {len(endpoints)} endpoints.")

Found 263 endpoints.


In [8]:
import asyncio
from playwright.async_api import async_playwright

# 1. Start Playwright and launch browser
# Leaving variables here so they stay in your local scope
p = await async_playwright().start()
browser = await p.chromium.launch(headless=True)
context = await browser.new_context(
    user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
)
page = await context.new_page()

# 2. Define the test URL
test_url = "https://site.financialmodelingprep.com/developer/docs/stable/search-name"

print(f"Navigating to {test_url}...")
await page.goto(test_url, wait_until="domcontentloaded", timeout=30000)

# 3. Wait for the actual UI to render (H1 is the baseline)
try:
    await page.wait_for_selector('h1', timeout=10000)
    # Give the JS an extra second to populate the tables/JSON blocks
    await asyncio.sleep(2) 
except Exception as e:
    print(f"H1 didn't load. Might be a bot block. Error: {e}")

# 4. Try to grab the data using a broad selector
# This looks for the main article or the div containing the word 'Endpoint'
# which is usually a safe anchor for documentation pages.
content_element = await page.query_selector('main') or await page.query_selector('article') or await page.query_selector('div[class*="singleDocument"]')

if content_element:
    raw_text = await content_element.inner_text()
    print("SUCCESS: Content captured.")
    print("-" * 30)
    print(raw_text[:500]) # Preview first 500 chars
    print("-" * 30)
else:
    print("FAILURE: Content element not found.")
    # Save the HTML so you can open it in a browser and see what's wrong
    html_snapshot = await page.content()
    with open("debug_test.html", "w", encoding="utf-8") as f:
        f.write(html_snapshot)
    print("Saved HTML to debug_test.html")

# 5. DO NOT close the browser yet so you can inspect variables in the next cell
# If you want to clean up later, run: await browser.close(); await p.stop()

Navigating to https://site.financialmodelingprep.com/developer/docs/stable/search-name...
SUCCESS: Content captured.
------------------------------
Home 
API Documentation 
Company Name Search API
Company Name Search API

Search for ticker symbols, company names, and exchange details for equity securities and ETFs listed on various exchanges with the FMP Name Search API. This endpoint is useful for retrieving ticker symbols when you know the full or partial company or asset name but not the symbol identifier.

Explore Endpoint
About Company Name Search API

The FMP Name Search API provides an easy way to find the ticker symbols and exchange
------------------------------


In [39]:
# 1. Identify the specific elements by class (using partial matches for robustness)
# Title
title_el = await page.query_selector('h1[class*="singleDocument_title"]')

# Subtitle / Intro
subtitle_el = await page.query_selector('div[class*="singleDocument_subTitle"]')

# Sections (This catches About, Features, Parameters, and Response)
# We use query_selector_all because there are usually multiple divs with this class
section_els = await page.query_selector_all('div[class*="singleDocument_section"]')

# 2. Extract and Filter
extracted_parts = []

if title_el:
    extracted_parts.append(await title_el.inner_text())

if subtitle_el:
    extracted_parts.append(await subtitle_el.inner_text())

for section in section_els:
    # Get the class attribute to check for exclusions
    class_attr = await section.get_attribute('class') or ""
    
    # EXCLUSION LOGIC: 
    # Skip the "How it works" section (works__4nX9k)
    # Also skip sections that usually contain "Related APIs" or "FAQs" if they share the same base class
    if 'singleDocument_works' in class_attr:
        continue
    
    # Additional check: If the text inside the section starts with unwanted headers, skip it
    section_text = await section.inner_text()
    unwanted_headers = ["Related", "FAQs"]
    if any(section_text.startswith(header) for header in unwanted_headers):
        continue
        
    extracted_parts.append(section_text)

# 3. Combine into one clean string
clean_content = "\n\n---\n\n".join(extracted_parts)

# 4. Preview result
print("CLEAN EXTRACTION COMPLETE")
print(f"Captured {len(extracted_parts)} segments.")
print("-" * 30)
print(clean_content[:1000]) # First 1000 chars of the clean version

CLEAN EXTRACTION COMPLETE
Captured 6 segments.
------------------------------
Company Name Search API

---

Home 
API Documentation 
Company Name Search API
Company Name Search API

Search for ticker symbols, company names, and exchange details for equity securities and ETFs listed on various exchanges with the FMP Name Search API. This endpoint is useful for retrieving ticker symbols when you know the full or partial company or asset name but not the symbol identifier.

Explore Endpoint

---

About Company Name Search API

The FMP Name Search API provides an easy way to find the ticker symbols and exchange information for companies and ETFs. This endpoint is useful for retrieving ticker symbols when you know the company or asset name but not the symbol identifier.

Key Features of the Name Search API

Simple Company Name Lookup: Enter a company or asset name, and retrieve the corresponding ticker symbol, company name, and exchange details.
Equity Securities and ETFs: Supports searches

In [41]:
from rich import print as rprint
rprint(clean_content)

Company Name Search API

---

Home 
API Documentation 
Company Name Search API
Company Name Search API

Search for ticker symbols, company names, and exchange details for equity securities and ETFs listed on various 
exchanges with the FMP Name Search API. This endpoint is useful for retrieving ticker symbols when you know the 
full or partial company or asset name but not the symbol identifier.

Explore Endpoint

---

About Company Name Search API

The FMP Name Search API provides an easy way to find the ticker symbols and exchange information for companies and 
ETFs. This endpoint is useful for retrieving ticker symbols when you know the company or asset name but not the 
symbol identifier.

Key Features of the Name Search API

Simple Company Name Lookup: Enter a company or asset name, and retrieve the corresponding ticker symbol, company 
name, and exchange details.
Equity Securities and ETFs: Supports searches for a variety of listed equity securities and exchange-traded funds 
(ETFs) across major exchanges.
Accurate and Up-to-Date Data: Receive real-time, accurate search results, ensuring you're always working with the 
latest available information.

How Investors and Analysts Can Benefit

Quick Symbol Lookup: Easily locate ticker symbols when you know the company name but not the corresponding symbol.
Broad Market Coverage: Search across multiple exchanges for both domestic and international companies, helping you 
stay informed about different markets.
Streamlined Workflow: Enhance your research and investment decisions by quickly identifying the correct symbols for
analysis or trade execution.

Endpoint:

https://financialmodelingprep.com/stable/search-name?query=AA

Company Name Search API Parameters

Query Parameter

Type

Example

query*

string

AA

limit

number

50

exchange

string

NASDAQ

(*) Required

Response
Code
Export
1
2
3
4
5
6
7
8
9

[
        {
                "symbol": "AAGUSD",
                "name": "AAG USD",
                "currency": "USD",
                "exchangeFullName": "CCC",
                "exchange": "CRYPTO"
        }
]

---

How it works

Get Started: Sign Up Today!

Begin your data journey by signing up and accessing our API endpoints. Get instant access to a vast array of 
financial data to power your applications and analyses.

Dive into Data: Free Plan Access

Explore our free data plan and access a wide range of financial data through our API endpoints. Start integrating 
real-time data into your applications and projects.

Unlock Premium Data: Upgrade Now!

Upgrade to our premium plan for exclusive access to advanced financial datasets via our API endpoints. Take your 
analyses to the next level and gain a competitive edge in the market.

---

Company Name Search API FAQs

How often is the data in the stock-screener API updated?

The screener API is updated in real time during market trading hours.

How can I search for company details using the CIK?

You can search for company details using the CIK by accessing our CIK search API. Simply input the CIK of the 
company you are looking for, and you will receive the relevant data. For more information and a sample of the data 
returned using the CIK search API, you can visit our documentation here:

Can I retrieve the 15 largest market cap stocks for each sector using the company-screener endpoint?

Currently, you can add the market cap parameter to the company-screener endpoint to filter stocks based on a 
specific market cap value, but the data will not be returned in sorted order from largest to smallest.

Can I search for stocks listed on international exchanges?

Yes, the API supports searches across global exchanges, making it easy to find where a stock is listed worldwide.

How can traders benefit from the Exchange Variants API?

Traders can use this API to identify all the exchanges where a stock is listed, helping them to understand the 
stock&#39;s liquidity and trading patterns across different markets.

How can I ret

In [ ]:
async def scrape_dynamic_content(browser, url):
    context = await browser.new_context(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    page = await context.new_page()

    try:
        await page.goto(url, wait_until="networkidle", timeout=60000)
        await page.wait_for_selector('h1[class*="singleDocument_title"]', timeout=15000)
        await asyncio.sleep(2)

        clean_text = await page.evaluate('''() => {
            const results = [];

            const ignoreClasses = ['singleDocument_works', 'singleDocument_faq', 'related'];
            const ignoreHeaders = ["How it works", "Unlock Premium", "Related", "FAQs"];

            function shouldIgnore(el) {
                const cls = el.className || "";
                const text = (el.innerText || "").trim();
                if (ignoreClasses.some(c => cls.includes(c))) return true;
                if (ignoreHeaders.some(h => text.startsWith(h))) return true;
                return false;
            }

            // --- Pass 1: Top-level non-section elements ---
            const topSelectors = [
                'h1[class*="singleDocument_title"]',
                'div[class*="singleDocument_subTitle"]',
            ];
            topSelectors.forEach(sel => {
                document.querySelectorAll(sel).forEach(el => {
                    const text = (el.innerText || "").trim();
                    if (text) results.push(text);
                });
            });

            // --- Pass 2: Walk sections, but extract innerHTML deeply ---
            const sections = document.querySelectorAll('div[class*="singleDocument_section"]');
            sections.forEach(section => {
                if (shouldIgnore(section)) return;

                // Instead of grabbing section.innerText (which flattens everything),
                // walk direct children so we preserve structural boundaries
                // and don't accidentally skip nested container_divs.
                const children = section.querySelectorAll(
                    'div[class*="container_div"], table, pre, code, p, h2, h3, h4, ul, ol'
                );

                if (children.length === 0) {
                    // Leaf section with no structured children — just grab text
                    const text = (section.innerText || "").trim();
                    if (text) results.push(text);
                    return;
                }

                const sectionParts = [];
                const seen = new Set();

                children.forEach(child => {
                    // Skip if an ancestor of this child is also in our children NodeList
                    // (avoids double-counting e.g. a <p> inside a container_div we already have)
                    let ancestor = child.parentElement;
                    let dominated = false;
                    while (ancestor && ancestor !== section) {
                        if (seen.has(ancestor)) { dominated = true; break; }
                        ancestor = ancestor.parentElement;
                    }
                    if (dominated) return;

                    seen.add(child);
                    const text = (child.innerText || "").trim();
                    if (text) sectionParts.push(text);
                });

                if (sectionParts.length > 0) {
                    results.push(sectionParts.join("\\n"));
                }
            });

            return results.join("\\n\\n---\\n\\n");
        }''')

        return clean_text

    except Exception as e:
        return f"Error scraping {url}: {str(e)}"
    finally:
        await context.close()

In [36]:
async def scrape_dynamic_content(browser, url):
    context = await browser.new_context(
        user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    )
    page = await context.new_page()

    try:
        await page.goto(url, wait_until="networkidle", timeout=20000)
        await page.wait_for_selector('h1[class*="singleDocument_title"]', timeout=5000)
        await asyncio.sleep(2)

        clean_text = await page.evaluate('''() => {
            const results = [];

            const ignoreClasses = [
                'singleDocument_works',
                'singleDocument_faq',
                'related'
            ];
            const ignoreHeaders = ["How it works", "Unlock Premium"];

            // Title and subtitle — simple, no nesting issues
            const title = document.querySelector('h1[class*="singleDocument_title"]');
            if (title) results.push(title.innerText.trim());

            const subtitle = document.querySelector('div[class*="singleDocument_subTitle"]');
            if (subtitle) results.push(subtitle.innerText.trim());

            // For sections: just grab the root section element and innerText the whole thing.
            // innerText already handles all nested children recursively — no need to walk manually.
            const sections = document.querySelectorAll('div[class*="singleDocument_section"]');
            sections.forEach(section => {
                const cls = section.className || "";
                const text = (section.innerText || "").trim();

                if (ignoreClasses.some(c => cls.includes(c))) return;
                if (ignoreHeaders.some(h => text.startsWith(h))) return;

  
                const parent = section.parentElement;

                if (text) results.push(text);
            });

            return results.join("\\n\\n---\\n\\n");
        }''')

        return clean_text

    except Exception as e:
        return f"Error scraping {url}: {str(e)}"
    finally:
        await context.close()

In [37]:
async def run_parallel_scrape(endpoint_list, max_concurrent=5):
    semaphore = asyncio.Semaphore(max_concurrent)
    results = []

    async with async_playwright() as p:
        # Launch one browser, but open multiple contexts (pages) within it
        browser = await p.chromium.launch(headless=True)

        async def sem_task(ep):
            async with semaphore:
                print(f"Starting: {ep['api_name']}")
                content = await scrape_dynamic_content(browser, ep['full_url'])
                return {
                    "api_name": ep['api_name'],
                    "url": ep['full_url'],
                    "content": content
                }

        # Create tasks for all endpoints
        tasks = [sem_task(ep) for ep in endpoint_list]
        
        # Gather results as they complete
        results = await asyncio.gather(*tasks)
        
        await browser.close()
    return results

# EXECUTION
# Adjust the slice endpoints[:10] to test a small batch first
final_data = await run_parallel_scrape(endpoints[:5], max_concurrent=5)

print(f"\nFinished! Scraped {len(final_data)} endpoints.")

Starting: Stock Symbol Search API
Starting: Company Name Search API
Starting: CIK API
Starting: CUSIP API
Starting: ISIN API

Finished! Scraped 5 endpoints.


In [38]:
from rich import print as rprint
rprint(final_data[0]['content'])

Stock Symbol Search API

---

Home 
API Documentation 
Stock Symbol Search API
Stock Symbol Search API

Easily find the ticker symbol of any stock with the FMP Stock Symbol Search API. Search by symbol across multiple 
global markets.

Explore Endpoint

---

About Stock Symbol Search API

The FMP Stock Symbol Search API allows users to quickly and efficiently locate stock ticker symbols. Whether you're
searching for U.S. stocks, international equities, or ETFs, this API provides fast, reliable results. Key features 
include:

Simple Search: Enter a company name or ticker symbol to retrieve essential details like the symbol, company name, 
exchange, and currency.
Global Market Access: Search across major stock exchanges, including NASDAQ, NYSE, and more.
Accurate and Up-to-Date: The API delivers real-time results, ensuring you're always working with the latest ticker 
information.

The Stock Symbol Search API is perfect for traders, investors, or anyone needing quick access to stock symbols 
across different markets.

Endpoint:

https://financialmodelingprep.com/stable/search-symbol?query=AAPL

Stock Symbol Search API Parameters

---

Stock Symbol Search API FAQs